In [15]:
import json
import os
import xml.etree.ElementTree as ET
from datetime import datetime

import httpx
import yaml
from dotenv import load_dotenv

found_env = load_dotenv("../../env_files/ncbi.env")
assert found_env == True, "Failed to find the relevant environment variable"

In [16]:
save_date = datetime.now().strftime("%b%Y")

In [17]:
QUERY = '"breast neoplasms"[mh] AND 2019:2026[dp] AND english[la] AND hasabstract NOT "retracted publication"[pt]'
query_config = {
    "db": "pubmed",
    "term": QUERY,
    "retmax": 5000,
    "sort": "pub_date",
    "retmode": "json",
}
with open("../configs/default_query.yaml", "w") as fp:
    yaml.safe_dump(query_config, fp, default_flow_style=False)

In [18]:
with open("../configs/default_query.yaml", "r") as fp:
    config = yaml.safe_load(fp)

In [19]:
BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
AUTH = {
    "email": os.environ["NCBI_EMAIL"],
    "api_key": os.environ.get("NCBI_API_KEY", ""),
}
QUERY = '"breast neoplasms"[mh] AND 2019:2026[dp] AND english[la] AND hasabstract NOT "retracted publication"[pt]'

with httpx.Client(timeout=60) as c:
    r = c.get(
        f"{BASE}/esearch.fcgi",
        params={**AUTH, **query_config},
    )
    pmids = sorted(r.json()["esearchresult"]["idlist"], key=int)

    with open(f"../data/corpus_{save_date}.jsonl", "w") as out:
        for i in range(0, len(pmids), 200):
            r = c.post(
                f"{BASE}/efetch.fcgi",
                data={
                    **AUTH,
                    "db": "pubmed",
                    "id": ",".join(pmids[i : i + 200]),
                    "retmode": "xml",
                },
            )
            for art in ET.fromstring(r.content).iter("PubmedArticle"):
                abstract = "\n".join(
                    (f"{a.get('Label')}: " if a.get("Label") else "")
                    + "".join(a.itertext())
                    for a in art.iter("AbstractText")
                )
                if not abstract:
                    continue
                out.write(
                    json.dumps(
                        {
                            "pmid": art.findtext(".//MedlineCitation/PMID"),
                            "title": "".join(art.find(".//ArticleTitle").itertext()),
                            "abstract": abstract,
                            "journal": art.findtext(".//Journal/Title"),
                            "pub_year": art.findtext(".//JournalIssue/PubDate/Year"),
                            "pub_types": [p.text for p in art.iter("PublicationType")],
                            "mesh_terms": [d.text for d in art.iter("DescriptorName")],
                            "doi": art.findtext(
                                ".//PubmedData/ArticleIdList/ArticleId[@IdType='doi']"
                            ),
                        }
                    )
                    + "\n"
                )

KeyboardInterrupt: 

In [20]:
with httpx.Client(timeout=60) as c:
    r = c.get(
        f"{BASE}/esearch.fcgi",
        params={**AUTH, **query_config},
    )
    pmids = sorted(r.json()["esearchresult"]["idlist"], key=int)
    i = 0
    r = c.post(
        f"{BASE}/efetch.fcgi",
        data={
            **AUTH,
            "db": "pubmed",
            "id": ",".join(pmids[i : i + 200]),
            "retmode": "xml",
        },
    )
    print(r.content)

b'<?xml version="1.0" ?>\n<!DOCTYPE PubmedArticleSet PUBLIC "-//NLM//DTD PubMedArticle, 1st January 2025//EN" "https://dtd.nlm.nih.gov/ncbi/pubmed/out/pubmed_250101.dtd">\n<PubmedArticleSet>\n<PubmedArticle><MedlineCitation Status="MEDLINE" Owner="NLM" IndexingMethod="Curated"><PMID Version="1">39625825</PMID><DateCompleted><Year>2026</Year><Month>06</Month><Day>17</Day></DateCompleted><DateRevised><Year>2026</Year><Month>08</Month><Day>13</Day></DateRevised><Article PubModel="Print-Electronic"><Journal><ISSN IssnType="Electronic">1538-9804</ISSN><JournalIssue CitedMedium="Internet"><Volume>49</Volume><Issue>4</Issue><PubDate><MedlineDate>2026 Jul-Aug 01</MedlineDate></PubDate></JournalIssue><Title>Cancer nursing</Title><ISOAbbreviation>Cancer Nurs</ISOAbbreviation></Journal><ArticleTitle>Psychological Symptom Cluster Severity Profiles Are Associated With Higher Levels of Stress and Worse Quality of Life in Postmenopausal Women With Breast Cancer.</ArticleTitle><Pagination><StartPage>E

In [ ]:
with open("data/corpus_Oct2026.jsonl", encoding="utf-8") as fp:
    corpus = [json.loads(line) for line in fp if line.strip()]

JSONDecodeError: Extra data: line 2 column 1 (char 2551)

In [26]:
art = next(ET.fromstring(r.content).iter("PubmedArticle"))
art.findtext(".//MedlineCitation/PMID")

'39625825'